# పాఠం 11 - ఏజెంట్-టు-ఏజెంట్ (A2A) ప్రోటోకాల్


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ప్రოტోకాల్ అంటే ఏమిటి?

The **Agent-to-Agent (A2A) protocol** ఒక ఓపెన్ ప్రమాణం, ఇది AI ఏజెంట్లకు పరస్పరం సంభాషించడానికి, ఒకరినొకరు కనుగొనడానికి, మరియు కలిసి పని చేసుకోవడానికి అనుమతిస్తుంది — అవి విభిన్న ఫ్రేమ్‌వర్క్‌లపై నిర్మించబడ్డప్పటికీ లేదా వేరే సేవలచే హోస్ట్ చేయబడ్డప్పటికీ కూడా.

Key concepts:

- **Discovery** – ఏజెంట్లు తమ సామర్థ్యాలను వివరించే *Agent Card*ని ప్రచురిస్తారు, తద్వారా ఇతర ఏజెంట్లు (లేదా ఆర్కెస్ట్రేటర్లు) ఒక పనికి సరిపడిన నిపుణునిని సులభంగా కనుగొనగలుగుతారు.
- **Message Passing** – ఏజెంట్లు సాధారణ ప్రోటోకాల్ ద్వారా నిర్మిత సంకలిత సందేశాలను మార్పిడి చేస్తారు, అందువల్ల ఒక ఏజెంట్ నుండి వచ్చిన అభ్యర్థనను మరో ఏజెంట్ అతని అంతర్గత అమలు విధానం ఎలాగైనా ఉన్నా కూడా అర్థం చేసుకుని నెరవేర్చగలదు.
- **Task Lifecycle** – A2A *submitted*, *working*, *completed*, and *failed* వంటి స్థితులను నిర్వచిస్తుంది, తద్వారా ఆర్కెస్ట్రేటర్‌కు అప్పగించబడిన పని ఎలా పురోగమిస్తున్నదనిపై పూర్తి పారదర్శకత కలుగుతుంది.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## ప్రత్యేకీకృత ప్రయాణ ఏజెంట్లను సృష్టించడం


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## వర్క్‌ఫ్లో ద్వారా బహుళ-ఏజెంట్ సహకారం

మేము ఈ మూడు ఏజెంట్లను A2A సందేశ పంపిణీని ప్రతిబింబించే క్రమమైన వర్క్‌ఫ్లోగా అనుసంధానిస్తాము:

1. **CurrencyExchangeAgent** వినియోగదారు అభ్యర్థనను స్వీకరించి కరెన్సీ మార్పిడి మార్గదర్శకాన్ని తయారుచేస్తుంది.
2. **ActivityPlannerAgent** సమృద్ధి చేసిన సందర్భాన్ని స్వీకరించి కార్యకలాప సూచనలను జోడిస్తుంది.
3. **TravelManagerAgent** రెండు ఇన్‌పుట్‌లను సమగ్రంగా సమన్వయించి చివరి ప్రయాణ సారాంశంగా తయారుచేస్తుంది.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ఉత్పత్తిలో A2A ను అర్థం చేసుకోవడం

ఉత్పత్తి పరిసరాల్లో A2A ప్రోటోకాల్ శక్తివంతమైన క్రాస్-సర్వీస్ పరిస్థితులను అనుమతిస్తుంది:

| Capability | Description |
|---|---|
| **ఫ్రేమ్‌వర్క్‌ల మధ్య ఇంటరోపరబిలిటీ** | ఒక ఫ్రేమ్‌వర్క్‌తో నిర్మించిన ఏజెంట్, A2A-అనుగుణమైన ఇతర ఏదైనా ఫ్రేమ్‌వర్క్‌తో నిర్మించిన ఏజెంట్‌కు పనులను అప్పగించగలదు, తద్వారా సంస్థల అంతులా నిజమైన ఇంటరోపరబిలిటీ సాధ్యమవుతుంది. |
| **సేవా సరిహద్దులు** | ఏజెంట్లు వేర్వేరు మైక్రోసర్వీసుల్లో, క్లౌడ్ ప్రాంతాల్లో లేదా పూర్తిగా వేరే సంస్థల్లో ఉండవచ్చు అయినప్పటికీ సజావుగా కలసికట్టుగా పని చేయగలవు. |
| **డైనమిక్ అన్వేషణ** | ఒక ఆర్కెస్ట్రేటర్ రన్‌టైమ్‌లో Agent Card రిజిస్ట్రీని ప్రశ్నించి ఇచ్చిన ఉప-పనికి అత్యుత్తమ నిపుణుడిని కనుగొనవచ్చు. |
| **స్ట్రీమింగ్ & పుష్ నోటిఫికేషన్లు** | A2A రియల్-టైమ్ ప్రగతి నవీకరణల కోసం Server-Sent Events (SSE) ను మరియు దీర్ఘకాలిక పనుల కోసం పుష్ నోటిఫికేషన్లను మద్దతు ఇస్తుంది. |

మేము పైగా నిర్మించిన వర్క్‌ఫ్లో ఈ ప్యాటర్న్ యొక్క సరళీకృత, ఇన్-ప్రోసెస్ వెర్షన్ మాత్రమే. వాస్తవ డిప్లాయ్‌మెంట్‌లో ప్రతి ఏజెంట్ ఒక HTTP ఎండ్‌పాయింట్‌ను ఎక్స్‌పోజ్ చేయగలదు, Agent Card ను ప్రచురిస్తాడు, మరియు A2A JSON-RPC ప్రోటోకాల్ ద్వారా సంభాషిస్తుంది.


## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నారు:

1. **A2A ప్రోటోకాల్ ఏమిటి** — ఏజెంట్-టు-ఏజెంట్ కనుగొనుట, సందేశాల మార్పిడి,
   మరియు పనుల నిర్వహణ కోసం ఒక ఓపెన్ స్టాండర్డ్.
2. **ఎలా ప్రత్యేక ఏజెంట్లను సృష్టించాలి** — ఒక కరెన్సీ ఎక్స్‌చేంజ్ ఏజెంట్, ఒక కార్యకలాపాల ప్లానర్ ఏజెంట్, మరియు ఒక ట్రావెల్ మేనేజర్ ఆర్కెస్ట్రేటర్.
3. **ఏజెంట్లను వర్క్‌ఫ్లోలో ఎలా కనెక్ట్ చేయాలి** — `WorkflowBuilder` ఉపయోగించి ఏజెంట్ల మధ్య క్రమంగా జరిగే
   సందేశాల మార్పిడిని నమూనా చేయడం.
4. **A2A ఉత్పత్తి వాతావరణంలో ఎలా పనిచేస్తుంది** — డైనమిక్ డిస్కవరీ మరియు స్ట్రీమింగ్ అప్‌డేట్లతో క్రాస్-ఫ్రేమ్‌వర్క్, క్రాస్-సర్వీస్ సహకారాన్ని సాధ్యం చేయడం.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
నివారణ:
ఈ డాక్యుమెంట్‌ను AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, ఆటోమెటెడ్ అనువాదాల్లో తప్పులు లేదా అపరిశుద్ధతలు ఉండే అవకాశం ఉంది అని దయచేసి గమనించండి. మూల భాషలోని అసలు డాక్యుమెంట్‌ను అధికారిక మూలంగా పరిగణించాలి. అత్యవసరమైన లేదా కీలక సమాచారం కోసం ప్రొఫెషనల్ మానవ అనువాదాన్ని సూచిస్తున్నాము. ఈ అనువాదం ఉపయోగం వలన కలిగిన ఏవైనా అపార్థాలు లేదా తప్పుగా అర్థం చేసుకోవడంపై మేము బాధ్యులేమి కావు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
